# SAM 3 (Lite/Text) Fine-Tuning Pipeline for Hieroglyphs

This notebook demonstrates the complete end-to-end pipeline to fine-tune **SAM 3**. 
**Features included:**
- **Explicit SAM 3 Classes**: Uses `Sam3Model` and `Sam3Processor` as per official docs.
- **Full Metrics**: Tracks Mean IoU and Pixel Accuracy for Train & Val.
- **Early Stopping**: Automated halt (Patience 3) based on Val Loss.
- **Final Test Pass**: Standalone evaluation cell at the end.

In [ ]:
# Install from source for SAM 3 architecture support
!pip uninstall -y transformers
!pip install git+https://github.com/huggingface/transformers.git accelerate pycocotools -q

In [ ]:
import os
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from glob import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Sam3Processor, Sam3Model
from pycocotools import mask as mask_util
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Metrics and Preprocessing

In [ ]:
def compute_iou(preds, targets):
    t = (targets > 0.5).float()
    if preds.shape[0] != t.shape[0]: t = t[:preds.shape[0]]
    if preds.shape[-2:] != t.shape[-2:]:
        t = torch.nn.functional.interpolate(t.unsqueeze(1), size=preds.shape[-2:], mode='nearest').squeeze(1)
    
    p_binary = (torch.sigmoid(preds) > 0.5).float()
    if p_binary.dim() == 4 and t.dim() == 3: p_binary = p_binary.squeeze(1)
    
    intersection = (p_binary * t).sum(dim=(1, 2))
    union = (p_binary + t).clamp(0, 1).sum(dim=(1, 2))
    return ((intersection + 1e-6) / (union + 1e-6)).mean().item()

def compute_pixel_accuracy(preds, targets):
    t = (targets > 0.5).float()
    if preds.shape[0] != t.shape[0]: t = t[:preds.shape[0]]
    if preds.shape[-2:] != t.shape[-2:]:
        t = torch.nn.functional.interpolate(t.unsqueeze(1), size=preds.shape[-2:], mode='nearest').squeeze(1)
    
    p_binary = (torch.sigmoid(preds) > 0.5).float()
    if p_binary.dim() == 4 and t.dim() == 3: p_binary = p_binary.squeeze(1)
    
    return (p_binary == t).float().mean().item()

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    # REFINED: Stronger median blur (size 5) to kill salt-and-pepper noise seen in sample
    img = cv2.medianBlur(img, 5)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2RGB))

## 2. Dataset Setup

In [ ]:
class HieroglyphsDataset(Dataset):
    def __init__(self, data_dir, processor):
        self.data_dir = data_dir
        self.processor = processor
        self.json_paths = sorted(glob(os.path.join(data_dir, "*.json")))
        
    def __len__(self):
        return len(self.json_paths)
    
    def __getitem__(self, idx):
        json_path = self.json_paths[idx]
        img_path = json_path.replace(".json", ".jpg")
        image = preprocess_image(img_path)
        if image is None: return None
        
        with open(json_path, 'r') as f: data = json.load(f)
        bboxes, gt_masks = [], []
        for ann in data['annotations']:
            if 'segmentation' not in ann: continue
            x, y, w, h = ann['bbox']
            bboxes.append([x, y, x + w, y + h])
            segm = ann['segmentation']
            if isinstance(segm['counts'], str): segm['counts'] = segm['counts'].encode('utf-8')
            # Strict binarization
            mask = (mask_util.decode(segm) > 0).astype(np.float32)
            gt_masks.append(mask)
        
        if not bboxes: return None
        MAX_SYMBOLS = 200
        bboxes = bboxes[:MAX_SYMBOLS]
        gt_masks = gt_masks[:MAX_SYMBOLS]
        
        inputs = self.processor(image, text="symbol", input_boxes=[bboxes], return_tensors="pt")
        inputs["gt_masks"] = torch.tensor(np.array(gt_masks), dtype=torch.float32)
        return {k: v.squeeze(0) if k != "gt_masks" else v for k, v in inputs.items()}

def custom_collate(batch):
    return [b for b in batch if b is not None]

MODEL_ID = "facebook/sam3"
processor = Sam3Processor.from_pretrained(MODEL_ID)
BASE_PATH = "/kaggle/input/datasets/karimtawfik/hieroglyphics-segmentation-data-sam-2-format/segmentation.v1-2023-03-12-7-33pm.sam2/"

train_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"train", processor), batch_size=2, shuffle=True, collate_fn=custom_collate)
valid_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"valid", processor), batch_size=2, shuffle=False, collate_fn=custom_collate)
test_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"test", processor), batch_size=1, shuffle=False, collate_fn=custom_collate)

## 3. Training Logic (Dice + BCE Hybrid)

In [ ]:
model = Sam3Model.from_pretrained(MODEL_ID).to(device)
# UNFREEZING REFINEMENT: Unfreeze the Mask Decoder AND the Neck
for name, param in model.named_parameters():
    if any(x in name for x in ["mask_decoder", "neck"]):
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
bce_loss = nn.BCEWithLogitsLoss()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred = pred.reshape(-1)
        target = target.reshape(-1)
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice

dice_loss_fn = DiceLoss()

def calc_loss(pred, target):
    # Strict Binarization sanity
    t = (target > 0.5).float()
    t = t.unsqueeze(1) if t.dim() == 3 else t
    if t.shape[0] > pred.shape[0]: t = t[:pred.shape[0]]
    t_resized = torch.nn.functional.interpolate(t, size=(pred.shape[-2], pred.shape[-1]), mode='nearest')
    if pred.dim() == 3: t_resized = t_resized.squeeze(1)
    
    bce = bce_loss(pred, t_resized)
    dice = dice_loss_fn(pred, t_resized)
    # HYBRID WEIGHTING: 80% Dice ensure symbols are found, 20% BCE keeps gradients stable
    return 0.2 * bce + 0.8 * dice

best_val_loss, patience, p_counter = float('inf'), 3, 0

for epoch in range(10):
    model.train()
    t_losses, t_ready = [], []
    t_ious, t_accs = [], []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")
    for batch in pbar:
        if not batch: continue
        batch_l = 0
        for item in batch:
            model_inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "gt_masks"}
            gt = item["gt_masks"].to(device)
            out = model(**model_inputs, multimask_output=False)
            
            count = min(gt.shape[0], out.pred_masks.squeeze(0).shape[0])
            pred = out.pred_masks.squeeze(0)[:count]
            gt = gt[:count]
            
            loss = calc_loss(pred, gt)
            batch_l += loss
            t_ious.append(compute_iou(pred.detach(), gt))
            t_accs.append(compute_pixel_accuracy(pred.detach(), gt))
            
        optimizer.zero_grad()
        batch_l.backward()
        optimizer.step()
        t_losses.append(batch_l.item() / len(batch))
        pbar.set_postfix({"loss": f"{np.mean(t_losses):.4f}", "iou": f"{np.mean(t_ious):.4f}", "acc": f"{np.mean(t_accs):.4f}"})

    model.eval()
    v_losses, v_ious, v_accs = [], [], []
    pbar_v = tqdm(valid_loader, desc=f"Epoch {epoch+1} [VALID]")
    with torch.no_grad():
        for batch in pbar_v:
            if not batch: continue
            for item in batch:
                model_inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "gt_masks"}
                gt = item["gt_masks"].to(device)
                out = model(**model_inputs, multimask_output=False)
                count = min(gt.shape[0], out.pred_masks.squeeze(0).shape[0])
                pred = out.pred_masks.squeeze(0)[:count]
                
                v_loss_val = calc_loss(pred, gt[:count]) 
                v_losses.append(v_loss_val.item())
                v_ious.append(compute_iou(pred, gt[:count]))
                v_accs.append(compute_pixel_accuracy(pred, gt[:count]))
            pbar_v.set_postfix({"v_loss": f"{np.mean(v_losses):.4f}", "v_iou": f"{np.mean(v_ious):.4f}", "v_acc": f"{np.mean(v_accs):.4f}"})

    avg_tl, avg_vl = np.mean(t_losses), np.mean(v_losses)
    print(f"Summary: T-Loss: {avg_tl:.4f} | T-IoU: {np.mean(t_ious):.4f} | T-Acc: {np.mean(t_accs):.4f} | V-Loss: {avg_vl:.4f} | V-IoU: {np.mean(v_ious):.4f} | V-Acc: {np.mean(v_accs):.4f}")

    if avg_vl < best_val_loss:
        best_val_loss, p_counter = avg_vl, 0
        model.save_pretrained("/kaggle/working/best_sam3_model")
        processor.save_pretrained("/kaggle/working/best_sam3_model")
        print("--> New Best SAM 3 Model Saved")
    else:
        p_counter += 1
        if p_counter >= patience: break

## 4. Final Evaluation Pass

In [ ]:
print("Evaluating on TEST set...")
best_model = Sam3Model.from_pretrained("/kaggle/working/best_sam3_model").to(device)
test_ious, test_accs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        if not batch: continue
        item = batch[0]
        model_inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() if k != "gt_masks"}
        out = best_model(**model_inputs, multimask_output=False)
        count = min(item["gt_masks"].shape[0], out.pred_masks.squeeze(0).shape[0])
        pred = out.pred_masks.squeeze(0)[:count]
        test_ious.append(compute_iou(pred.cpu(), item["gt_masks"][:count]))
        test_accs.append(compute_pixel_accuracy(pred.cpu(), item["gt_masks"][:count]))

print(f"FINAL RESULTS | Test mIoU: {np.mean(test_ious):.4f} | Test Pixel Accuracy: {np.mean(test_accs):.4f}")